In [ ]:
# Jupyter notebook for Session 0.
# NB! Restart kernel and clear all outputs before committing to git.

from dotenv import load_dotenv
import os
import sqlalchemy as sa
import pandas as pd
import plotly.express as px
from supabase import create_client
import kaleido # For saving Plotly figures as files.
from pathlib import Path # Filesystem tools.

In [ ]:
# Load configuration from .env file in parent directory.
# dotenv is smart enough to search whole directory tree for .env files.
load_dotenv(override=True)

# Connect to Supabase directly.
supabase_direct = sa.create_engine(os.getenv("SUPABASE_CONNECTION_STRING"))

# And via API, for demonstration purposes.
supabase_api = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Loading data using SQL query over direct connection.
# More flexible control over sizes of data sets.
def load_data_using_sql(sql):
    query = sa.text(sql)
    return pd.read_sql(query, supabase_direct)

# Loading data over supabase API.
# Should not be done in real-world situations when working with tables with millions of rows!
# Actually, Supabase API has built-in limit of 1000 rows - so this method could also yield misleading results.
def load_data_using_api(table):
    response = supabase_api.table(table).select('*').execute()
    return pd.DataFrame(response.data)


In [ ]:
# Load all data from sales table.
df = load_data_using_sql("SELECT * FROM sales")
df

In [ ]:
# 5 first rows from sales table.
df.head()

In [ ]:
# number of rows x number of columns
df.shape

In [ ]:
# Number of rows in sales table.
df.shape[0]

In [ ]:
# Generate descriptive statistics.
df.describe()

In [ ]:
# Columns, data types, count of NOT NULL values.
df.info()

In [ ]:
# Let's define a dataset.
data = {
    'customer_id': [1001, 1002, 1003, 1001, 1002, 1004, 1003, 1001, 1005, 1004,
                    1002, 1003, 1005, 1001, 1006, 1004, 1002, 1007, 1003, 1005],
    'sale_date': ['2024-01-15', '2024-01-16', '2024-02-01', '2024-02-20', '2024-03-01',
                   '2024-03-05', '2024-03-15', '2024-04-10', '2024-04-12', '2024-04-20',
                   '2024-05-01', '2024-05-10', '2024-05-15', '2024-06-01', '2024-06-05',
                   '2024-06-10', '2024-06-20', '2024-07-01', '2024-07-05', '2024-07-10'],
    'total_price': [89.99, 45.50, 120.00, 67.30, 55.00, 210.00, 33.50, 145.00, 78.00, 92.00,
                     160.00, 44.00, 88.50, 230.00, 37.00, 175.00, 110.00, 65.00, 95.00, 125.00],
    'city': ['Tallinn', 'Tartu', 'Tallinn', 'Tallinn', 'Tartu', 'Pärnu', 'Tallinn', 'Tallinn',
             'Tartu', 'Pärnu', 'Tartu', 'Tallinn', 'Tartu', 'Tallinn', 'Pärnu', 'Pärnu',
             'Tartu', 'Tallinn', 'Tallinn', 'Tartu'],
    'product_category': ['Dresses', 'Tops', 'Denim', 'Accessories', 'Tops', 'Denim', 'Tops',
                         'Dresses', 'Denim', 'Accessories', 'Dresses', 'Tops', 'Denim',
                         'Dresses', 'Accessories', 'Denim', 'Tops', 'Accessories', 'Dresses', 'Denim']
}

# Constructing a pandas DataFrame using the given data set.
df = pd.DataFrame(data)

In [ ]:
# Number of rows and columns.
df.shape

In [ ]:
# 5 first rows
df.head()

In [ ]:
# Basic info.
df.info()

In [ ]:
# Statistics.
df.describe()

In [ ]:
# Data types of columns.
df.dtypes

In [ ]:
# Number of unique customer ids.
df["customer_id"].nunique()

In [ ]:
# Unique city names in the data frame.
df["city"].unique()

In [ ]:
# Number of occurrences of each city in the data frame.
df["city"].value_counts()

In [ ]:
# Total revenue - sum of total price of each sale.
df["total_price"].sum()

In [ ]:
# Number of unique product categories. Similar to count(*) in PostgreSQL.
df["product_category"].nunique()

In [ ]:
# Unique product categories. Similar to SELECT DISTINCT in PostgreSQL.
df["product_category"].unique()

In [ ]:
# Count of values of each unique product category.
# Similar to: SELECT product_category, count(*) AS products_count FROM products GROUP BY product_category;
df["product_category"].value_counts()

In [ ]:
# Descriptive statistics for product_category column.
df["product_category"].describe()

In [ ]:
# Boolean indexing.
df["total_price"] > 100

In [ ]:
# Using boolean indexing to filter the data frame to include only rows where total_price is greater than 100.
# In SQL, it would be done like this: SELECT * FROM sales WHERE total_price > 100;
df[df["total_price"] > 100]

In [ ]:
# Filtered data frame where city is Tallinn in each row.
# SQL filter: WHERE city = 'Tallinn'
df[df["city"] == "Tallinn"]

In [ ]:
# Filtering pandas DataFrame by total price and city.
# SQL filter: WHERE total_price > 100 AND city = 'Tallinn'.
df[(df["total_price"] > 100) & (df["city"] == "Tallinn")]

In [ ]:
# Summarizing total revenue for each city.
# In SQL: use GROUP BY.
grouped = df.groupby("city")["total_price"].sum()
print(type(grouped))
grouped

In [ ]:
# Using multiple aggregate functions over a column.
grouped = df.groupby('city')['total_price'].agg(['sum', 'mean', 'count'])
print(type(grouped))
grouped

In [ ]:
# Note that agg(['sum']) behaves differently from sum(). It returns a DataFrame object instead of a Series.
grouped = df.groupby('city')['total_price'].agg(['sum'])
print(type(grouped))
grouped

In [ ]:
# Summarizing over multiple columns.
df.groupby(["city", "product_category"])["total_price"].sum()

In [ ]:
# Multiple aggregations over multiple columns.
grouped = df.groupby(["city", "product_category"])["total_price"].agg(["sum", "mean", "count", "min", "max"])

In [ ]:
# Sorting by total price in descending order.
# SQL: ORDER BY total_price DESC;
df.sort_values('total_price', ascending=False)

In [ ]:
# Sorting by multiple columns.
# SQL: ORDER BY city ASC, total_price DESC.
df.sort_values(['city', 'total_price'], ascending=[True, False])

In [ ]:
df_sales = load_data_using_sql("SELECT * FROM sales;")
df_customers = load_data_using_sql("SELECT * FROM customers;")

In [ ]:
# Joining pandas data frames.
# SQL:
"""
SELECT
    *
FROM sales s
LEFT JOIN customers c ON s.customer_id = s.customer_id
"""
merged = pd.merge(df_sales, df_customers, on='customer_id', how='left')
print(type(merged))
merged.head()

In [ ]:
# Let's check shapes.
# Number of rows in merged should be the same as df_sales.
# Number of columns in merged should be one less than the number of columns in df_sales plus df_customers,
# because customer_id column will not be duplicated in merged dataframe.
df_sales.shape, df_customers.shape, merged.shape

In [ ]:
# Adding a new column to pandas data frame.
df['discount'] = df['total_price'] * 0.1
df.head()

In [ ]:
# Conditional calculations on total_price column to add another column (segment).
# SQL: CASE WHEN.
df['segment'] = df['total_price'].apply(
    lambda x: 'Big' if x > 100 else 'Small'
)
df.head()

In [ ]:
# Converting sale dates to datetime.
df['sale_date'] = pd.to_datetime(df['sale_date'])
df.head()

In [ ]:
# Extracting month and year from sale date and storing them in extra columns.
df["month"] = df["sale_date"].dt.month
df["year"] = df["sale_date"].dt.year
df.head()

In [ ]:
df_sales.head()

In [ ]:
# Adding a new column to merged dataframe which adds business interpretation to total price.
merged["order_size"] = merged["total_price"].apply(
    lambda x: "Suur (100+)" if x >= 100 else "Väike (<100)"
)
merged.head()

In [ ]:
# All sales rows where customer city is Tallinn.
tallinn = merged[merged['city'] == 'Tallinn']
tallinn.describe()

In [ ]:
# Aggregations over all cities with cities with largest sum appearing at the top.
city_revenue = merged.groupby('city')['total_price'].agg(['sum', 'mean', 'count']).sort_values("sum", ascending=False)
# Rename columns to provide better business context:
city_revenue.columns = ["total_revenue", "average_revenue", "orders"]
city_revenue.head()

In [ ]:
# Statistics on aggregations (which is statistics by itself)
city_revenue.describe()

In [ ]:
order_sizes_series = merged['order_size'].value_counts()
print(type(order_sizes_series))

# reset_index() will convert Series to DataFrame:
order_sizes = order_sizes_series.reset_index()
print(type(order_sizes))

order_sizes

In [ ]:
# Top customer cities by average revenue.
city_revenue.sort_values("average_revenue", ascending=False).head(10)

In [ ]:
# Named Aggregation - naming aggregations in a different way than using `by_customer.columns =`
customer_summary = merged.groupby(["customer_id", "first_name", "last_name"]).agg(
    total_spending=("total_price", "sum"),
    average_order=("total_price", "mean"),
    orders=("total_price", "count")
).sort_values("total_spending", ascending=False).reset_index()

# Adding VIP status.
customer_summary["vip_status"] = customer_summary["total_spending"].apply(
    lambda x: "YES" if x > 200 else "NO"    # Täida lüngad!
)

# TOP 5 customers with id and name, by total spending.
customer_summary.head()

In [ ]:
# Find out how many VIP customers there are.
customer_summary["vip_status"].value_counts().reset_index()

In [ ]:
# Bar chart of total revenue by city.
city_data = merged.groupby("city")["total_price"].sum().reset_index()
fig = px.bar(
    city_data,
    x="city",
    y="total_price",
    title="UrbanStyle käive linnade kaupa",
    labels={"city": "Linn", "total_price": "Käive"}
)

# Customizing layout.
fig.update_layout(
    plot_bgcolor='white',
    font=dict(family="Calibri", size=14),
    title_font_size=18
)

# Customizing axes.
fig.update_xaxes(title_text="Linn", tickangle=45)
fig.update_yaxes(title_text="Käive (EUR)")

# Saving as HTML.
Path('../tmp').mkdir(exist_ok=True) # Create a directory for temporary files (should already be in .gitignore).
fig.write_html("../tmp/urbanstyle_chart.html")

# Saving as image (kaleido package is used internally by Plotly for that)
fig.write_image("../tmp/urbanstyle_chart.png")

# SVG (Scalable Vector Graphics) is also supported.
# No pixellation when zooming in an SVG image.
fig.write_image("../tmp/urbanstyle_chart.svg")

fig.show()

In [ ]:
# Line chart of total revenue by month.
monthly = merged.groupby(merged["sale_date"].dt.to_period("M"))["total_price"].sum().reset_index()
monthly["sale_date"] = monthly["sale_date"].astype(str)

fig = px.line(
    monthly,
    x="sale_date",
    y="total_price",
    title="UrbanStyle kuukäibe trend",
    labels={"sale_date": "Kuu", "total_price": "Käive (EUR)"},
    markers=True # Visualize data points as bold dots on the chart
)
fig.show()

In [ ]:
customer_summary.head()

In [ ]:
# Scatter plot.
# Customer purchase patterns: number of orders vs total spending.
# Kliendi ostukäitumine: tellimuste arv vs kogukulutus
fig = px.scatter(
    customer_summary,
    x="orders",
    y="total_spending",
    color="vip_status",
    title="Kliendid: ostusagedus vs kogukulutus",
    labels={ "orders": "Tellimuste arv", "total_spending": "Kogukulutus (EUR)", "vip_status": "VIP staatus" }
)
fig.show()

In [ ]:
df.head()

In [ ]:
# Pie chart.
# Percent of total revenue by product category.
cat_data = df.groupby("product_category")["total_price"].sum().reset_index()
fig = px.pie(
    cat_data,
    values="total_price",
    names="product_category",
    title="Käibe jaotus tootekategooriate kaupa",
    labels={ "total_price": "Käive", "product_category": "Tootekategooria" }
)
fig.show()

In [ ]:
# Revenue by product category.
cat_revenue = df.groupby("product_category")["total_price"].sum().reset_index()
cat_revenue = cat_revenue.sort_values("total_price", ascending=True)

fig = px.bar(
    cat_revenue,
    x="total_price",
    y="product_category",
    orientation="h",
    title="UrbanStyle: Käive tootekategooriate kaupa",
    labels={
        "total_price": "Käive (EUR)",
        "product_category": "Kategooria"
    },
    text="total_price", # Display total prices on bars
    color="product_category"
)

fig.update_layout(
    title="Käive tootekategooriate lõikes",
    showlegend=False # Hide legend
)

fig.show()

In [ ]:
# Convert sale dates to monthly periods.
months = merged["sale_date"].dt.to_period("M")
months

In [ ]:
# groupby links indexes of merged and months internally - that's why the following code works.
monthly_sales = merged.groupby(months).agg(total_revenue=("total_price", "sum")).reset_index()
# Add a stringified month field, because Period type would cause an error with Plotly.
monthly_sales["month"] = monthly_sales["sale_date"].astype(str)
monthly_sales.head()

In [ ]:
fig = px.line(
    monthly_sales,
    title="Kogukäive kuude lõikes",
    x="month",
    y="total_revenue",
    labels={ "month": "Kuu", "total_revenue": "Kogukäive (EUR)" },
    markers=True
)

fig.update_layout(
    plot_bgcolor="white"
)

fig.show()

In [ ]:
customer_summary.head()

In [ ]:
fig = px.pie(
    customer_summary,
    title="Klientide jaotus VIP-staatuse järgi",
    names="vip_status",
    labels={ "vip_status": "VIP-staatus" }
)

fig.update_layout(
    showlegend=True
)

fig.show()

In [ ]:
# RFM analysis.

# Reference date.
reference_date = pd.to_datetime("2024-12-31")

# All sales until the reference date.
df = merged[merged["sale_date"] <= reference_date]
df.info()

In [ ]:
# Recency (R): days since last purchase.
recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (reference_date - recency['last_purchase']).dt.days
recency.head()

In [ ]:
# Frequency (F): number of purchases
frequency = df.groupby("customer_id").size().reset_index(name="frequency")
frequency.head()

In [ ]:
# Monetary (M): total spending
monetary = df.groupby("customer_id")["total_price"].sum().reset_index()
monetary.columns = ["customer_id", "monetary"]
monetary.head()

In [ ]:
# Calculating all RFM metrics at once - much shorter and simpler and no further merging needed.
rfm = df.groupby("customer_id").agg(
    last_purchase=("sale_date", "max"), # recency
    number_of_purchases=("id", "size"), # frequency - size behaves like PostgreSQL count(*)
    total_spending=("total_price", "sum") # monetary
).reset_index()
rfm["time_since_last_purchase"] = reference_date - rfm["last_purchase"]
rfm["days_since_last_purchase"] = rfm["time_since_last_purchase"].dt.days
rfm.head()

In [ ]:
pd.qcut?

In [ ]:
# Settings for RFM score calculations.
q = 3 # number of quantiles
labels = list(range(1, q + 1)) # labels for F and M
reverse_labels = list(range(q, 0, -1)) # labels for R - smaller values of days_since_last_purchase are better
labels, reverse_labels # print out both

In [ ]:
# Recency score.
rfm["R_score"] = pd.qcut(rfm["days_since_last_purchase"], q, labels=reverse_labels).astype(int)
rfm.head()

In [ ]:
# Frequency score.
rfm["F_score"] = pd.qcut(rfm["number_of_purchases"], q, labels).astype(int)
rfm.head()

In [ ]:
# Monerary score.
rfm["M_score"] = pd.qcut(rfm["total_spending"], q, labels).astype(int)
rfm.head()

In [ ]:
# Summary RFM score.
rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]
rfm.head()

In [ ]:
# Segmentation.
def assign_segment(score):
    if score >= 8:
        return "VIP Champions"
    elif score >= 6:
        return "Loyal Customers"
    elif score >= 4:
        return "Potential Loyalists"
    else:
        return "At Risk"

rfm["segment"] = rfm["RFM_score"].apply(assign_segment)
rfm.sort_values("RFM_score", ascending=False)

In [ ]:
# Count customers by segment.
segment_counts = rfm.groupby("segment").agg(
    customers=("customer_id", "size"),
    total_spending=("total_spending", "sum")
).reset_index()
segment_counts

In [ ]:
# Visualize customers by segment, using bar chart.
fig1 = px.bar(
    segment_counts,
    x="segment",
    y="customers",
    title="UrbanStyle: Kliendisegmentide jaotus (RFM)",
    labels={"segment": "Segment", "customers": "Klientide arv"},
    color="segment"
)
fig1.show()

In [ ]:
# Visualize days since last purchase vs total spending.
fig2 = px.scatter(
    rfm,
    x="days_since_last_purchase",
    y="total_spending",
    color="segment",
    size="number_of_purchases", # bubble size on scatter plot
    hover_data=["customer_id"],
    title="UrbanStyle: Recency vs Monetary (RFM)",
    labels={
        "days_since_last_purchase": "Päevi viimasest ostust",
        "total_spending": "Kogukulutus (EUR)"
    }
)

fig2.update_layout(
    plot_bgcolor="white"
)

fig2.show()

In [ ]:
fig3 = px.pie(
    segment_counts,
    title="Klientide jaotus VIP-staatuse järgi",
    names="segment",
    values="total_spending",
    labels={ "segment": "Segment", "total_spending": "Kogukulutused" }
)

fig3.update_layout(
    showlegend=True
)

fig3.show()